# Patient-level random-effects models and supplementary figures

Run all cells from the revision_analysis directory. The notebook reads the supplied patient contrasts, fits cohort and pooled effects, repeats the fits after omitting each cohort, and draws Figures S15–S17. It writes to `03_result/06_public_reproduction/` or to `06_public_reproduction/` under `PLOS_REVISION_DIR`. See the README for the separate workflow starting from full count matrices.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
from importlib import import_module

import pandas as pd
from IPython.display import Image, display

# Read the released patient tables. The next cell fits new models.
base = Path.cwd()
if not (base / "analysis/meta_patient_inputs.csv").is_file():
    raise FileNotFoundError("Run this notebook from revision_analysis.")
sys.path.insert(0, str(base / "code"))
patient = import_module("00a_patient_expression")
result_dir = patient.result_root() / "06_public_reproduction"
subprocess_env = {**os.environ, "PLOS_REVISION_DIR": str(result_dir.resolve())}
if (result_dir / "01_counts_and_preprocessing/09_cohort_cell_cache").exists():
    raise ValueError("This directory contains full-count results. Choose a separate directory for the public run.")
model_outputs = {"meta_cohort_effects", "meta_summary", "meta_leave_one_out"}
for name, filename in patient.table_files.items():
    if name in model_outputs:
        continue
    target = result_dir / filename
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(base / "analysis" / (name + ".csv"), target)

input_table = pd.read_csv(result_dir / patient.table_files["meta_patient_inputs"])
print("Patient contrasts:", len(input_table), "Models:", input_table["analysis"].nunique())

## Fit random-effects models

Each patient contributes one expression contrast to a model. Pool the cohort means and their sampling variances with REML, using modified Knapp–Hartung confidence intervals.

In [ ]:
# Fit the models in R, then select the primary and uniform annotation results by name.
rscript = os.environ.get("RSCRIPT") or shutil.which("Rscript")
if not rscript:
    raise FileNotFoundError("Install R with metafor and put Rscript on PATH, or set RSCRIPT.")
fit = subprocess.run(
    [rscript, str(base / "code/04_fit_random_effects_models.R"), str(result_dir)],
    env=subprocess_env, capture_output=True, text=True
)
log = result_dir / "04_random_effects_models/05_r_model_log.txt"
log.parent.mkdir(parents=True, exist_ok=True)
log.write_text(fit.stdout + fit.stderr, encoding="utf-8")
fit.check_returncode()
model_table = pd.read_csv(result_dir / patient.table_files["meta_summary"])
baseline_table = model_table.loc[model_table["analysis"].isin(["primary_labels", "uniform_labels"])]
fields = ["analysis", "k", "n", "estimate", "ci_low", "ci_high", "p_value", "tau2", "I2"]
display(baseline_table[fields + ["prediction_low", "prediction_high"]])

## Check fitted effects against the reference tables

SciPy independently fits the REML model using the cohort means and variances. Compare every column in the three model tables after aligning results by analysis, cohort and omitted cohort, as applicable.

In [ ]:
# Check the models in the current Python environment using the released source-count audit.
subprocess.run(
    [sys.executable, str(base / "code/05a_validate_patient_and_model_results.py")],
    env=subprocess_env, check=True
)
keys = {
    "meta_summary": ["analysis"], "meta_cohort_effects": ["analysis", "cohort"],
    "meta_leave_one_out": ["analysis", "omitted"],
}
for name, columns in keys.items():
    reference_table = pd.read_csv(base / "analysis" / (name + ".csv"))
    observed_table = pd.read_csv(result_dir / patient.table_files[name])
    reference_table = reference_table.set_index(columns, verify_integrity=True).sort_index()
    observed_table = observed_table.set_index(columns, verify_integrity=True).sort_index()
    pd.testing.assert_frame_equal(reference_table, observed_table, check_dtype=False, rtol=1e-6, atol=1e-8)
print("Models checked:", len(model_table))

## Draw supplementary figures

S15 shows cohort and pooled effects. S16 compares normalization and cell-filtering choices. S17 shows each GSE162631 patient's cell counts, detection fractions and source-count contrasts.

In [ ]:
subprocess.run([sys.executable, str(base / "code/05b_plot_supplementary_figures.py")], env=subprocess_env, check=True)
figure_dir = result_dir / "05_supplementary_outputs"
for name in ["02_s15_random_effects_forest_plot", "03_s16_preprocessing_comparison", "04_s17_source_count_audit"]:
    display(Image(filename=str(figure_dir / (name + ".png"))))

## Interpret the fitted estimates

The confidence interval describes uncertainty in the pooled mean. The prediction interval describes the range of effects expected in a comparable cohort. The output also reports how often the confidence interval includes zero after one cohort is omitted.

In [ ]:
baseline_model_table = baseline_table.set_index("analysis", verify_integrity=True)
original = baseline_model_table.loc["primary_labels"]
uniform = baseline_model_table.loc["uniform_labels"]
prediction_crosses_zero = uniform.prediction_low <= 0 <= uniform.prediction_high
omission_table = pd.read_csv(result_dir / patient.table_files["meta_leave_one_out"])
uniform_omission_table = omission_table.loc[omission_table["analysis"] == "uniform_labels"]
crosses_zero = (uniform_omission_table["ci_low"] <= 0) & (uniform_omission_table["ci_high"] >= 0)
print(f"Pooled estimate with primary labels: {original.estimate:.3f}.")
status = "includes zero" if prediction_crosses_zero else "excludes zero"
print(f"Uniform-label prediction interval [{uniform.prediction_low:.3f}, {uniform.prediction_high:.3f}] {status}.")
print(f"Leave-one-cohort-out CIs including zero: {int(crosses_zero.sum())}/{len(crosses_zero)}.")